In [ ]:
!pip install ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 83.1 MB/s eta 0:00:00


In [26]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from datetime import datetime

# =========================
# BASE DE DATOS (MEMORIA)
# =========================

pacientes = []
medicos = []
citas = []
consultas = []

# =========================
# LISTAS
# =========================

TIPOS_SANGRE = ["O+", "O-", "A+", "A-", "B+", "B-", "AB+", "AB-"]
REGIMENES = ["CONTRIBUTIVO", "SUBSIDIADO"]

ESPECIALIDADES = [
    "Medicina General", "Pediatría", "Cardiología",
    "Ortopedia", "Dermatología", "Neurología", "Psiquiatría"
]

CONSULTORIOS = ["A1", "A2", "A3", "B1", "B2", "B3"]
HORARIOS = ["08:00-20:00", "20:00-08:00"]

# =========================
# VALIDACIONES
# =========================

def validar_nombre(change):
    if any(char.isdigit() for char in change['new']):
        nombre.value = ''.join(filter(lambda x: not x.isdigit(), change['new']))

def validar_fecha(texto):
    try:
        datetime.strptime(texto, "%Y/%m/%d")
        return True
    except:
        return False

# =========================
# PACIENTES
# =========================

def vista_pacientes():
    clear_output()

    global nombre

    doc = widgets.Text(description="Documento:")
    nombre = widgets.Text(description="Nombre:")
    nombre.observe(validar_nombre, names='value')

    fecha = widgets.Text(description="Fecha nacimiento:")
    sangre = widgets.Dropdown(options=TIPOS_SANGRE, description="Sangre:")
    eps = widgets.Text(description="EPS:")
    regimen = widgets.Dropdown(options=REGIMENES, description="Régimen:")
    ant = widgets.Text(description="Antecedentes:")

    btn = widgets.Button(description="Guardar")
    btn_volver_menu = widgets.Button(description="Volver al menú")

    def guardar(b):
        if not validar_fecha(fecha.value):
            print("Fecha inválida (AAAA/MM/DD)")
            return

        pacientes.append({
            "doc": doc.value,
            "nombre": nombre.value,
            "fecha": fecha.value,
            "sangre": sangre.value,
            "eps": eps.value,
            "regimen": regimen.value,
            "ant": ant.value
        })

        print("✔ Paciente registrado")

    btn.on_click(guardar)
    btn_volver_menu.on_click(lambda b: (clear_output(), display(menu, btn_ir)))

    display(doc, nombre, fecha, sangre, eps, regimen, ant, btn, btn_volver_menu)

    # EDITAR
    if pacientes:
        docs = [p["doc"] for p in pacientes]

        selector = widgets.Dropdown(options=docs, description="Editar:")

        btn_edit = widgets.Button(description="Editar")

        def editar(b):
            p = next(x for x in pacientes if x["doc"] == selector.value)

            nombre.value = p["nombre"]
            fecha.value = p["fecha"]

            def guardar_edit(b):
                p["nombre"] = nombre.value
                p["fecha"] = fecha.value
                print("✔ Actualizado")

            btn.on_click(guardar_edit)

        btn_edit.on_click(editar)

        display(selector, btn_edit)

# =========================
# MÉDICOS
# =========================

def vista_medicos():
    clear_output()

    doc = widgets.Text(description="Registro:")
    nombre = widgets.Text(description="Nombre:")
    nombre.observe(validar_nombre, names='value')

    esp = widgets.Dropdown(options=ESPECIALIDADES, description="Especialidad:")
    cons = widgets.Dropdown(options=CONSULTORIOS, description="Consultorio:")
    horario = widgets.Dropdown(options=HORARIOS, description="Horario:")

    btn = widgets.Button(description="Guardar")
    btn_volver_menu = widgets.Button(description="Volver al menú")

    def guardar(b):
        medicos.append({
            "doc": doc.value,
            "nombre": nombre.value,
            "esp": esp.value,
            "cons": cons.value,
            "horario": horario.value
        })
        print("✔ Médico registrado")

    btn.on_click(guardar)
    btn_volver_menu.on_click(lambda b: (clear_output(), display(menu, btn_ir)))

    display(doc, nombre, esp, cons, horario, btn, btn_volver_menu)

    # FILTROS
    filtro_esp = widgets.Dropdown(options=ESPECIALIDADES, description="Filtrar Esp:")
    btn_filtro = widgets.Button(description="Buscar")

    def buscar(b):
        for m in medicos:
            if m["esp"] == filtro_esp.value:
                print(m)

    btn_filtro.on_click(buscar)

    display(filtro_esp, btn_filtro)

# =========================
# CITAS
# =========================

def generar_horas(horario):
    inicio, fin = horario.split("-")
    horas = []
    h_inicio = int(inicio[:2])
    h_fin = int(fin[:2])

    current_hour = h_inicio
    while True:
        horas.append(f"{current_hour:02d}:00")
        horas.append(f"{current_hour:02d}:30")

        current_hour += 1
        if current_hour == 24:
            current_hour = 0

        if current_hour == h_fin and h_fin != h_inicio:

            break
        elif h_fin == h_inicio and len(horas) > 0 and horas[-1] == f"{h_fin:02d}:30":
            break
        elif h_fin == h_inicio and h_inicio == h_inicio % 24 and len(horas) > 0 and current_hour == h_inicio:

            break

    return horas

def vista_citas():
    clear_output()

    if not pacientes or not medicos:
        print(" Debes registrar pacientes y médicos")
        btn_volver_menu_temp = widgets.Button(description="Volver al menú")
        btn_volver_menu_temp.on_click(lambda b: (clear_output(), display(menu, btn_ir)))
        display(btn_volver_menu_temp)
        return

    doc = widgets.Dropdown(options=[p["doc"] for p in pacientes], description="Paciente:")
    med = widgets.Dropdown(options=[m["doc"] for m in medicos], description="Médico:")

    fecha = widgets.Text(description="Fecha:")

    hora = widgets.Dropdown(options=[], description="Hora:")

    def actualizar_horas(change):
        m = next(x for x in medicos if x["doc"] == med.value)
        hora.options = generar_horas(m["horario"])

    med.observe(actualizar_horas, names='value')


    if medicos:
        actualizar_horas({'new': med.value})

    motivo = widgets.Text(description="Motivo:")
    btn = widgets.Button(description="Crear cita")
    btn_volver_menu = widgets.Button(description="Volver al menú")

    def guardar(b):
        cod = str(len(citas)+1).zfill(3)

        citas.append({
            "cod": cod,
            "doc": doc.value,
            "med": med.value,
            "fecha": fecha.value,
            "hora": hora.value,
            "motivo": motivo.value,
            "estado": "ACTIVA"
        })

        print("✔ Cita creada:", cod)

    btn.on_click(guardar)
    btn_volver_menu.on_click(lambda b: (clear_output(), display(menu, btn_ir)))

    display(doc, med, fecha, hora, motivo, btn, btn_volver_menu)

    # =========================
    # CONSULTAR Y CANCELAR CITAS
    # =========================

    if citas:
        cods_all = [c["cod"] for c in citas]
        if cods_all:
            consulta_output = widgets.Output()
            sel_consultar = widgets.Dropdown(options=cods_all, description="Cita a consultar:")
            btn_consultar = widgets.Button(description="Consultar Detalles")

            def on_consult_click(b):
                with consulta_output:
                    clear_output()
                    cita_to_consult = next((c for c in citas if c["cod"] == sel_consultar.value), None)
                    if cita_to_consult:
                        print("Detalles de la cita:")
                        for key, value in cita_to_consult.items():
                            print(f"- {key}: {value}")
                    else:
                        print("Cita no encontrada.")

            btn_consultar.on_click(on_consult_click)
            display(widgets.HBox([sel_consultar, btn_consultar]), consulta_output)

        cods_activos = [c["cod"] for c in citas if c["estado"] == "ACTIVA"]
        if cods_activos:
            display(widgets.HTML("<h3>Cancelar Cita</h3>"))

            sel_cancelar = widgets.Dropdown(options=cods_activos, description="Seleccione cita:")
            motivo_cancelacion = widgets.Text(description="Motivo de cancelación:", placeholder="Ingrese el motivo aquí")
            btn_confirmar_cancelacion = widgets.Button(description="Confirmar Cancelación")

            def on_confirm_cancel_click(cb):
                cita_to_cancel = next((c for c in citas if c["cod"] == sel_cancelar.value), None)
                if cita_to_cancel:
                    cita_to_cancel["estado"] = "CANCELADA"
                    cita_to_cancel["motivo_cancelacion"] = motivo_cancelacion.value
                    print(f"✔ Cita {sel_cancelar.value} cancelada. Motivo: {motivo_cancelacion.value}")
                    motivo_cancelacion.value = ""
                    vista_citas()
                else:
                    print("Error: Cita no encontrada o ya no está activa.")

            btn_confirmar_cancelacion.on_click(on_confirm_cancel_click)

            display(sel_cancelar, motivo_cancelacion, btn_confirmar_cancelacion)
        else:
            print("No hay citas activas para cancelar.")

    def update_filter_widgets(change):
        # =========================
        # LISTAR CITAS (ARREGLADO)
        # =========================

        display(widgets.HTML("<h3>Listar Citas</h3>"))

        dropdown_list_by = widgets.Dropdown(
            options=['Paciente', 'Doctor'],
            description='Listar por:'
        )

        sel_filter_id = widgets.Dropdown(description='Seleccionar:')
        fecha_inicio = widgets.Text(description='Fecha inicio:')
        fecha_fin = widgets.Text(description='Fecha fin:')
        btn_buscar = widgets.Button(description='Buscar citas')

        output_resultados = widgets.Output()

        # =========================
        # ACTUALIZAR LISTA SEGÚN FILTRO
        # =========================

        def actualizar_lista(change):
            if dropdown_list_by.value == "Paciente":
                sel_filter_id.options = [p["doc"] for p in pacientes]
            else:
                sel_filter_id.options = [m["doc"] for m in medicos]

        dropdown_list_by.observe(actualizar_lista, names='value')

        # Inicializar lista
        actualizar_lista(None)

        # =========================
        # FUNCIÓN DE BÚSQUEDA
        # =========================

        def buscar_citas(b):
            with output_resultados:
                clear_output()

                if not validar_fecha(fecha_inicio.value) or not validar_fecha(fecha_fin.value):
                    print("Fechas inválidas (AAAA/MM/DD)")
                    return

                inicio = datetime.strptime(fecha_inicio.value, "%Y/%m/%d")
                fin = datetime.strptime(fecha_fin.value, "%Y/%m/%d")

                encontrados = []

                for c in citas:
                    fecha_c = datetime.strptime(c["fecha"], "%Y/%m/%d")

                    if dropdown_list_by.value == "Paciente":
                        if c["doc"] == sel_filter_id.value and inicio <= fecha_c <= fin:
                            encontrados.append(c)
                    else:
                        if c["med"] == sel_filter_id.value and inicio <= fecha_c <= fin:
                            encontrados.append(c)

                if encontrados:
                    for c in encontrados:
                        print(c)
                else:
                    print("No hay resultados")

        btn_buscar.on_click(buscar_citas)

        # =========================
        # MOSTRAR INTERFAZ (UNA SOLA VEZ)
        # =========================

        display(
            dropdown_list_by,
            sel_filter_id,
            fecha_inicio,
            fecha_fin,
            btn_buscar,
            output_resultados
        )
# =========================
# REGISTRO CLÍNICO
# =========================

def vista_registro_clinico():
    clear_output()

    if not pacientes or not medicos:
        print("Debes registrar pacientes y médicos primero")
        return

    display(widgets.HTML("<h2>Registro Clínico</h2>"))

    # =========================
    # SELECCIÓN
    # =========================

    paciente = widgets.Dropdown(
        options=[(p["nombre"], p["doc"]) for p in pacientes],
        description="Paciente:"
    )

    medico = widgets.Dropdown(
        options=[(m["nombre"], m["doc"]) for m in medicos],
        description="Médico:"
    )

    # =========================
    # REGISTRO CONSULTA
    # =========================

    display(widgets.HTML("<h3>Registro Consulta</h3>"))

    diagnostico = widgets.Text(
        description="Diagnóstico:",
        placeholder="Ej: Gripe"
    )

    sintomas = widgets.Text(
        description="Síntomas:"
    )

    presion = widgets.Text(description="Presión:")
    temperatura = widgets.Text(description="Temperatura:")
    frecuencia = widgets.Text(description="Frecuencia cardíaca:")

    observaciones = widgets.Text(description="Observaciones:")

    # =========================
    # REGISTRO TRATAMIENTO
    # =========================

    display(widgets.HTML("<h3>Registro Tratamiento</h3>"))

    nombre_med = widgets.Text(description="Medicamento:")
    dosis = widgets.Text(description="Dosis:")
    frecuencia_med = widgets.Text(description="Frecuencia:")
    dias = widgets.IntText(description="Días:")

    btn_guardar = widgets.Button(description="Guardar consulta")
    output = widgets.Output()

    # =========================
    # GUARDAR
    # =========================

    def guardar(b):
        with output:
            clear_output()

            if diagnostico.value.strip() == "":
                print("Debes ingresar diagnóstico")
                return

            consulta = {
                "doc": paciente.value,
                "med": medico.value,
                "fecha": datetime.now().strftime("%Y/%m/%d"),

                "diagnostico": diagnostico.value + " (CIE-10)",
                "sintomas": sintomas.value,

                "signos": {
                    "presion": presion.value,
                    "temperatura": temperatura.value,
                    "frecuencia": frecuencia.value
                },

                "observaciones": observaciones.value,

                "medicamentos": [
                    {
                        "nombre": nombre_med.value,
                        "dosis": dosis.value,
                        "frecuencia": frecuencia_med.value,
                        "dias": dias.value
                    }
                ]
            }

            consultas.append(consulta)

            print("✔ Consulta registrada correctamente")

    btn_guardar.on_click(guardar)

    display(
        paciente, medico,
        diagnostico, sintomas,
        presion, temperatura, frecuencia,
        observaciones,
        nombre_med, dosis, frecuencia_med, dias,
        btn_guardar,
        output
    )

    # =========================
    # HISTORIAL CLÍNICO
    # =========================

    display(widgets.HTML("<h3>Consultar Historial Clínico</h3>"))

    paciente_hist = widgets.Dropdown(
        options=[(p["nombre"], p["doc"]) for p in pacientes],
        description="Paciente:"
    )

    btn_hist = widgets.Button(description="Ver historial")
    output_hist = widgets.Output()

    def ver_historial(b):
        with output_hist:
            clear_output()

            historial = [c for c in consultas if c["doc"] == paciente_hist.value]

            if not historial:
                print("No hay consultas")
                return

            # Orden cronológico
            historial.sort(key=lambda x: x["fecha"])

            for c in historial:
                print("-----")
                print("Fecha:", c["fecha"])
                print("Diagnóstico:", c["diagnostico"])
                print("Síntomas:", c["sintomas"])
                print("Signos:", c["signos"])
                print("Observaciones:", c["observaciones"])
                print("Medicamentos:")

                for m in c["medicamentos"]:
                    print(f"  - {m['nombre']} | {m['dosis']} | {m['frecuencia']} | {m['dias']} días")

    btn_hist.on_click(ver_historial)

    display(paciente_hist, btn_hist, output_hist)

    # =========================
    # VOLVER
    # =========================

    btn_volver = widgets.Button(description="Volver al menú")
    btn_volver.on_click(lambda b: (clear_output(), display(menu, btn_ir)))

    display(btn_volver)


# =========================
# ANALISIS DE DATOS
# =========================

def vista_analisis():
    clear_output()

    display(widgets.HTML("<h2>📊 Análisis de Datos</h2>"))

    # =========================
    # 1. DISTRIBUCIÓN EPS / RÉGIMEN
    # =========================

    btn_eps = widgets.Button(description="Ver distribución EPS y Régimen")
    output_eps = widgets.Output()

    def calcular_eps(b):
        with output_eps:
            clear_output()

            if not pacientes:
                print("No hay pacientes registrados")
                return

            conteo = {}

            for p in pacientes:
                clave = (p["eps"], p["regimen"])

                if clave not in conteo:
                    conteo[clave] = 0

                conteo[clave] += 1

            print("Distribución de pacientes:\n")

            for (eps, reg), cantidad in conteo.items():
                print(f"EPS: {eps} | Régimen: {reg} → {cantidad} pacientes")

    btn_eps.on_click(calcular_eps)

    display(btn_eps, output_eps)

    # =========================
    # 2. MEDICAMENTOS MÁS USADOS
    # =========================

    btn_meds = widgets.Button(description="Ver medicamentos más prescritos")
    output_meds = widgets.Output()

    def calcular_meds(b):
        with output_meds:
            clear_output()

            if not consultas:
                print("No hay consultas registradas")
                return

            conteo = {}

            for c in consultas:
                if "medicamentos" in c:
                    for m in c["medicamentos"]:
                        nombre = m["nombre"]

                        if nombre not in conteo:
                            conteo[nombre] = 0

                        conteo[nombre] += 1

            if conteo:
                print("Medicamentos más prescritos:\n")

                for med, cant in sorted(conteo.items(), key=lambda x: x[1], reverse=True):
                    print(f"{med} → {cant} veces")
            else:
                print("No hay medicamentos registrados")

    btn_meds.on_click(calcular_meds)

    display(btn_meds, output_meds)

    # =========================
    # BOTÓN VOLVER
    # =========================

    btn_volver = widgets.Button(description="Volver al menú")
    btn_volver.on_click(lambda b: (clear_output(), display(menu, btn_ir)))

    display(btn_volver)



# =========================
# HISTORIAL CLÍNICO
# =========================


# =========================
# MENÚ PRINCIPAL
# =========================

menu = widgets.Dropdown(
    options=[
        ("Pacientes", 1),
        ("Médicos", 2),
        ("Citas", 3),
        ("Registro Clínico", 4),
        ("Análisis", 5)

    ],
    description="Módulo:"
)

btn_ir = widgets.Button(description="Ir")

def navegar(b):
    if menu.value == 1:
        vista_pacientes()
    elif menu.value == 2:
        vista_medicos()
    elif menu.value == 3:
        vista_citas()
    elif menu.value == 4:
        vista_registro_clinico()
    elif menu.value == 5:
        vista_analisis()

btn_ir.on_click(navegar)

display(menu, btn_ir)


HTML(value='<h2>📊 Análisis de Datos</h2>')

Button(description='Ver distribución EPS y Régimen', style=ButtonStyle())

Output()

Button(description='Ver medicamentos más prescritos', style=ButtonStyle())

Output()

Button(description='Volver al menú', style=ButtonStyle())